In [ ]:
import torch
import numpy as np
import os
from PIL import Image, ImageFilter
from huggingface_hub import snapshot_download
from PIL import ImageEnhance
from diffusers import AutoPipelineForInpainting, LCMScheduler
import time
from IPython.display import display
import datetime
from diffusers import StableDiffusionInpaintPipeline, LCMScheduler
from skimage import color
import gc
import optimum
from optimum.intel import OVStableDiffusionInpaintPipeline
import cv2
#%%
import psutil
import os
import platform
from controlnet_aux import OpenposeDetector

#ADDITIONAL

파이썬 우선순위를 뒤로 미뤄놓는 코드.

성능이 좋다면 사용하지 않아도 됨.

In [ ]:

def optimize_my_system():
    # 현재 운영체제 확인 (윈도우에서만 작동하는 설정이 많음)
    if platform.system() == "Windows":
        try:
            # 1. 현재 실행 중인 내 파이썬 프로세스 가져오기
            current_p = psutil.Process(os.getpid())
            
            # 2. 우선순위를 '보통 이하(Below Normal)'로 설정
            # IDLE_PRIORITY_CLASS(낮음)보다 한 단계 위인 설정입니다.
            current_p.nice(psutil.BELOW_NORMAL_PRIORITY_CLASS)
            
            print("✅ 파이썬 우선순위가 '보통 이하'로 설정되었습니다.")
            
            # 3. (옵션) 크롬 우선순위 조정 - 필요 없으면 이 아래는 삭제해도 됩니다.
            # for proc in psutil.process_iter(['name']):
            #     try:
            #         if 'chrome.exe' in proc.info['name'].lower():
            #             # 크롬을 '보통 이상'으로 올려서 영상 끊김 방지
            #             proc.nice(psutil.ABOVE_NORMAL_PRIORITY_CLASS)
            #     except (psutil.NoSuchProcess, psutil.AccessDenied):
            #         continue
                    
        except Exception as e:
            print(f"❌ 설정 중 오류 발생: {e}")
    else:
        # 리눅스/맥 환경 (nice 값 10은 윈도우의 '보통 이하'와 유사함)
        os.nice(10)
        print("✅ 유닉스 계열 우선순위가 조정되었습니다.")

optimize_my_system()

In [ ]:
import os
import datetime
import torch
import numpy as np
import cv2
import gc
from PIL import Image, ImageFilter

# 🎯 [수정됨] LCM 스케줄러 임포트
from diffusers import LCMScheduler
from optimum.intel import OVStableDiffusionInpaintPipeline

# ==============================================================================
# 1. 환경 및 경로 설정
# ==============================================================================
LOCAL_MODEL_PATH = "./models/meinahentai_openvino"
ORIGINAL_IMAGE_PATH = "./original.jpg"
MASK1_PATH = "./mask1_updated.png"
MASK2_PATH = "./mask2_updated.png"
OUTPUT_IMAGE_PREFIX = "seethrough_lcm_only"


In [ ]:
def load_inpainting_pipeline():
    print("🧠 iGPU 가속 모델 로딩 중...")
    ov_config = {"CACHE_DIR": "ov_cache"}
    
    pipe = OVStableDiffusionInpaintPipeline.from_pretrained(
        LOCAL_MODEL_PATH, 
        export=False,          
        local_files_only=True,
        device="GPU",          
        ov_config=ov_config,
        torch_dtype=torch.float16,
        safety_checker=None,          
        feature_extractor=None,       
        requires_safety_checker=False 
    )
    
    print("⚡ 스케줄러를 LCM으로 변경 중...")
    pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)
    
    print("📏 모델 입력 규격 최적화 (512x512)...")
    pipe.reshape(batch_size=1, height=512, width=512, num_images_per_prompt=1)
    pipe.compile()
    
    return pipe

In [ ]:
def run_generation_multi_mask(pipe, idx):
    original_image = Image.open(ORIGINAL_IMAGE_PATH).convert("RGB")
    original_size = original_image.size
    final_output = original_image.copy()
    
    mask1_body = Image.open(MASK1_PATH).convert("L")        
    mask2_target = Image.open(MASK2_PATH).convert("L")   
    
    mask1_arr = np.array(mask1_body)
    non_zero_coords = cv2.findNonZero(mask1_arr)
    if non_zero_coords is None:
        return
        
    x, y, w, h = cv2.boundingRect(non_zero_coords)
    
    padding = 1
    left, top = max(0, x - padding), max(0, y - padding)
    right, bottom = min(original_size[0], x + w + padding), min(original_size[1], y + h + padding)
    max_side = max(right - left, bottom - top)
    cx, cy = left + (right - left) // 2, top + (bottom - top) // 2

    sq_left, sq_top = max(0, cx - max_side // 2), max(0, cy - max_side // 2)
    sq_right, sq_bottom = sq_left + max_side, sq_top + max_side

    if sq_left < 0: sq_right += (0 - sq_left); sq_left = 0
    if sq_top < 0: sq_bottom += (0 - sq_top); sq_top = 0
    if sq_right > original_size[0]: sq_left -= (sq_right - original_size[0]); sq_right = original_size[0]
    if sq_bottom > original_size[1]: sq_top -= (sq_bottom - original_size[1]); sq_bottom = original_size[1]
        
    sq_left, sq_top = max(0, sq_left), max(0, sq_top)
    sq_right, sq_bottom = min(original_size[0], sq_right), min(original_size[1], sq_bottom)
    
    crop_image = final_output.crop((sq_left, sq_top, sq_right, sq_bottom))
    crop_mask_target = mask2_target.crop((sq_left, sq_top, sq_right, sq_bottom))
    crop_mask_feathered = crop_mask_target.filter(ImageFilter.GaussianBlur(radius=1))

    crop_size = crop_image.size
    process_size = (512, 512)
    
    input_image = crop_image.resize(process_size, Image.Resampling.LANCZOS)
    input_mask  = crop_mask_target.resize(process_size, Image.Resampling.LANCZOS)

    # ==========================================================================
    # 🪄 파이프라인 실행
    # ==========================================================================
    
    print(f"   -> [1단계] 베이스 생성 중...")
    skin_result = pipe(
        prompt=PROMPT_SKIN,
        negative_prompt=NEG_PROMPT_SKIN,
        image=input_image,
        mask_image=input_mask,
        num_inference_steps=9,      
        guidance_scale=2,          
        strength=0.7              
    ).images[0]

    skin_path = f'./blending/step1_skin_{idx}.jpg'

    skin_result.save(skin_path)
    print(f'skin_image: {skin_path}')

    print(f"   -> [2단계] 원본 옷(색상 유지)과 50% 투명도로 겹치기...")

    # dyed_skin = ImageChops.multiply(skin_result, input_image)
    # blended_image = Image.blend(skin_result, dyed_skin, alpha=0.7)
    # blended_image = Image.blend(input_image, skin_result, alpha=0.7)

    # print(f"   -> [3단계] 반투명 이미지 위에 질감 코팅 중...")
    # final_result = pipe(
    #     prompt=PROMPT_SEE_THROUGH,
    #     negative_prompt=NEG_PROMPT_SEE_THROUGH,
    #     image=blended_image,         
    #     mask_image=input_mask,
    #     num_inference_steps=6,      
    #     guidance_scale=1.5,          
    #     strength=0.25                
    # ).images[0]
    
    # final_result = blended_image

    enhancer_contrast = ImageEnhance.Contrast(input_image)
    cloth_base = enhancer_contrast.enhance(1.3) 

    # 2. [1차 합성] 
    blended_image = Image.blend(skin_result, cloth_base, alpha=0.3)

    # 3. [원본 색상 복구 코팅] 
    final_result = Image.blend(blended_image, input_image, alpha=0.4)
    
    # ==========================================================================

    result = final_result.filter(ImageFilter.UnsharpMask(radius=1, percent=120, threshold=3))
    result_rescaled = result.resize(crop_size, Image.Resampling.LANCZOS)
    result_rgb = result_rescaled.convert("RGB")
    crop_mask_feathered_restored = crop_mask_feathered.resize(crop_size, Image.Resampling.LANCZOS)
    
    final_output.paste(result_rgb, (sq_left, sq_top), crop_mask_feathered_restored)

    os.makedirs("./outputs", exist_ok=True)
    now = datetime.datetime.now().strftime("%H%M%S")
    filename = f"./outputs/{OUTPUT_IMAGE_PREFIX}_{idx+1}_{now}.jpg"
    final_output.save(filename)
    print(f"저장 완료: {filename}")

In [ ]:
PROMPT_SKIN = (
    #prompt로 원하는 태그들
)
NEG_PROMPT_SKIN = (
    #없었으면 하는 태그들(amputee라든지)
)

In [ ]:
change_pipeline = False # 첫 실행시 true로 모델을 올려놓아야 함(openvino)

try:
    if change_pipeline:
        pipe = load_inpainting_pipeline()
    
    total_images = 5 #testing
    print(f"\n🚀 총 {total_images}장의 시스루 이미지 생성")

    for i in range(total_images):  
        print(f"\n🎨 [{i+1}/{total_images}] 번째 이미지 작업 시작...")
        
        run_generation_multi_mask(pipe, i)
        
        # 메모리 정리
        gc.collect()
            
    print(f"\n✅ 모든 작업 완료")

except Exception as e:
    print(f"\n❌ 실행 중 에러 발생: {e}")
    import traceback
    traceback.print_exc()